# Predict next ingredients order

This notebook converts aggregated sales forecasts into ingredient needs, then produces a simple “next order” table.

## Import & data loading

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# We use Exponential Gradient trick aggregation predictions from previous code
df = pd.read_csv("data_tmp/predictions_with_eg_3experts.csv")

# Final forecasts we'll ues to predict ingredient needs
final_forecasts = df[['Date', 'product','real_sale', 'eg_prediction']].rename(columns={'eg_prediction': 'final_forecast'})

## Building 'Ingredient consumption by day dataframe'

In [77]:
# Ingredients per product
ingredient_per_product = pd.read_csv("data_tmp/ingredients_per_unite.csv")

# List of all ingredients & all products UNIQUE
list_ingredient = ingredient_per_product.columns.tolist()
print(list_ingredient)
list_ingredient.remove('product') # Remove 'product' column from ingredient list

list_product = ingredient_per_product['product'].tolist()
list_product.remove('VIK_BREAD')
list_product = sorted(list_product)


#  Merge: ratios ingrédients/unités
df = final_forecasts.merge(ingredient_per_product, on="product", how="left")


# real_* et predict_*
for ing in list_ingredient:
    df[f"real_{ing}"] = df["real_sale"] * df[ing]
    df[f"predict_{ing}"] = df["final_forecast"] * df[ing]


# Bring back into final forecasts 
final_forecasts = df

final_forecasts.head()

['product', 'farine', 'sel', 'levure', 'sucre', 'beurre', 'lait', 'barres_chocolat', 'graines', 'farine_de_seigle', 'son_de_ble', 'creme', 'oeufs', 'chocolat']


,Date,product,real_sale,final_forecast,farine,sel,levure,sucre,beurre,lait,...,real_farine_de_seigle,predict_farine_de_seigle,real_son_de_ble,predict_son_de_ble,real_creme,predict_creme,real_oeufs,predict_oeufs,real_chocolat,predict_chocolat
0,2021-01-02,BAGUETTE,46.0,27.188576,0.16,0.0045,0.0005,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021-01-03,BAGUETTE,35.0,46.167079,0.16,0.0045,0.0005,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021-01-04,BAGUETTE,30.0,39.692664,0.16,0.0045,0.0005,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021-01-05,BAGUETTE,29.0,28.219830,0.16,0.0045,0.0005,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021-01-06,BAGUETTE,0.0,4.393184,0.16,0.0045,0.0005,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [78]:
ing_cols = [c for c in final_forecasts.columns if c.startswith("real_") or c.startswith("predict_")]
ing_cols.remove('real_sale')

daily_ingredients = (
    final_forecasts
    .groupby("Date", as_index=False)[ing_cols]
    .sum()
)

daily_ingredients.shape

(644, 27)

## Are our prediction any good ?

Reminder : 

Train: 2021-01-02 -> 2022-03-31 | n = 420

Val:   2022-04-01 -> 2022-06-30 | n = 90

Test:  2022-07-01 -> 2022-09-30 | n = 90

Prediction : 2022-10-01 -> 2022-10-07 | n = 7


In [79]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

real_cols = [c for c in final_forecasts.columns if c.startswith("real_")]
pairs = [(r, "predict_" + r[len("real_"):]) for r in real_cols
         if ("predict_" + r[len("real_"):]) in final_forecasts.columns]

test = final_forecasts[(final_forecasts["Date"] >= "2022-07-01") & (final_forecasts["Date"] <= "2022-09-30")]

y = np.concatenate([test[r].to_numpy() for r, p in pairs])
yhat = np.concatenate([test[p].to_numpy() for r, p in pairs])

mask = np.isfinite(y) & np.isfinite(yhat)   # enlève NaN et inf
rmse = np.sqrt(mean_squared_error(y[mask], yhat[mask]))
mae  = mean_absolute_error(y[mask], yhat[mask])

print(f"TEST RMSE (global ingrédients): {rmse:.4f}")
print(f"TEST MAE  (global ingrédients): {mae:.4f}")


TEST RMSE (global ingrédients): 0.4635
TEST MAE  (global ingrédients): 0.0883


## Final order for the next week

In [80]:
last_7_days = final_forecasts.sort_values("Date").tail(7)
order = {r.replace("real_", ""): float(last_7_days[p].sum()) for r, p in pairs}

order_df = (pd.DataFrame(order.items(), columns=["ingredient", "qty_pred_7days"])
              .sort_values("qty_pred_7days", ascending=False)
              .reset_index(drop=True))

display(order_df)

,ingredient,qty_pred_7days
0,farine,10.582916
1,farine_de_seigle,0.437034
2,sel,0.311327
3,beurre,0.156094
4,sucre,0.124875
5,creme,0.124875
6,oeufs,0.093656
7,levure,0.056906
8,lait,0.000000
9,barres_chocolat,0.000000
